# 03. Regresión Logística: Tu Primer Clasificador

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 60 minutos  
**Prerequisitos:** [01. Regresión Lineal](01-regresion-lineal.ipynb), [02. Gradient Descent](02-gradient-descent.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender la diferencia entre regresión y clasificación
- Derivar y aplicar la función sigmoide para probabilidades
- Implementar la función de pérdida cross-entropy
- Entrenar un clasificador binario desde cero
- Interpretar las predicciones como probabilidades
- Evaluar clasificadores usando métricas apropiadas (accuracy, precision, recall, F1)

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades personalizadas
import sys
sys.path.append('../../shared/utils')
from visualization import plot_decision_boundary, plot_confusion_matrix
from testing import test_exercise, check_shape, check_close
from datasets import load_dataset, generate_binary_classification

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Librerías importadas correctamente")

---
## 📌 1. Motivación: De Predecir Números a Predecir Categorías

### El Problema del Mundo Real

Eres científico de datos en un banco. Tu jefe te dice:

> *"Tenemos datos de 10,000 clientes. Necesito que predijas quién **incumplirá** su préstamo (sí/no) basándote en su historial crediticio, ingresos y deudas."*

Este NO es un problema de regresión (predecir un número continuo). Es un problema de **clasificación binaria**: predecir una de dos categorías.

### ¿Por qué no usar Regresión Lineal?

Imagina intentar predecir:
- ¿El paciente tiene cáncer? (Sí = 1, No = 0)

Con regresión lineal podrías obtener:
- Paciente A: predicción = 0.8 ✓ (sensato)
- Paciente B: predicción = -0.3 ❌ (¿probabilidad negativa?)
- Paciente C: predicción = 1.7 ❌ (¿más que 100% de probabilidad?)

**Necesitamos algo que siempre produzca valores entre 0 y 1.**

### ¿Por qué Regresión Logística?

- 🎯 **Diseñada específicamente para clasificación**
- 📊 **Las predicciones son probabilidades** (entre 0 y 1)
- 🧠 **Interpretable**: puedes entender el impacto de cada feature
- 🚀 **Base de redes neuronales**: la neurona sigmoide es regresión logística
- ⚡ **Rápida y eficiente**: funciona bien incluso con muchas features

### Aplicaciones Reales

- 🏥 Diagnóstico médico (enfermo vs sano)
- 💳 Detección de fraude (fraude vs legítimo)
- 📧 Clasificación de spam (spam vs no spam)
- 🎯 Marketing (comprará vs no comprará)
- 🔒 Autenticación (usuario legítimo vs impostor)

### La Pregunta Guía

> **¿Cómo convertir la salida de un modelo lineal en una probabilidad válida, y cómo entrenar el modelo para que esas probabilidades sean precisas?**

---
## 📊 2. Intuición Visual: La Función Sigmoide

El "truco" de la regresión logística es pasar la salida lineal por una función especial.

In [ ]:
# Definir la función sigmoide
def sigmoid(z):
    """Función sigmoide: σ(z) = 1 / (1 + e^(-z))"""
    return 1 / (1 + np.exp(-z))

# Crear valores de z (salida lineal)
z = np.linspace(-10, 10, 200)
sigma_z = sigmoid(z)

# Visualización
fig = go.Figure()

# Curva sigmoide
fig.add_trace(go.Scatter(
    x=z,
    y=sigma_z,
    mode='lines',
    name='σ(z) = 1/(1+e⁻ᶻ)',
    line=dict(color='blue', width=3)
))

# Líneas de referencia
fig.add_hline(y=0.5, line_dash="dash", line_color="red", 
              annotation_text="Umbral de decisión (0.5)")
fig.add_vline(x=0, line_dash="dash", line_color="green",
              annotation_text="z = 0")

# Regiones de clasificación
fig.add_vrect(x0=-10, x1=0, fillcolor="red", opacity=0.1,
              annotation_text="Clase 0", annotation_position="top left")
fig.add_vrect(x0=0, x1=10, fillcolor="green", opacity=0.1,
              annotation_text="Clase 1", annotation_position="top right")

fig.update_layout(
    title="La Función Sigmoide: Convirtiendo Números en Probabilidades",
    xaxis_title="z = w·x + b (salida lineal)",
    yaxis_title="σ(z) = probabilidad",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n💡 Propiedades Clave de la Sigmoide:")
print(f"   • Rango: siempre entre 0 y 1 (perfecto para probabilidades)")
print(f"   • σ(0) = 0.5 (punto de decisión)")
print(f"   • σ(∞) → 1 (alta confianza clase 1)")
print(f"   • σ(-∞) → 0 (alta confianza clase 0)")
print(f"   • Es una función suave y diferenciable (necesario para gradient descent)")

In [ ]:
# Visualización: Clasificación en 2D
# Generar datos sintéticos de clasificación binaria
X, y = generate_binary_classification(n_samples=200, n_features=2, random_state=42)

# Visualizar
fig = px.scatter(
    x=X[:, 0], y=X[:, 1], color=y.astype(str),
    labels={'x': 'Feature 1', 'y': 'Feature 2', 'color': 'Clase'},
    title="Problema de Clasificación Binaria",
    color_discrete_map={'0': 'red', '1': 'blue'}
)

fig.update_layout(
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n💡 Nuestro objetivo: encontrar una línea (frontera de decisión) que separe las clases.")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $y \in \{0, 1\}$ | Etiqueta binaria (clase) |
| $\hat{y} = P(y=1|x)$ | Probabilidad predicha de clase 1 |
| $\sigma(z)$ | Función sigmoide |
| $z = w^T x + b$ | Combinación lineal (logit) |
| $\mathcal{L}$ | Log-loss (Binary Cross-Entropy) |

### El Modelo de Regresión Logística

**Paso 1: Combinación lineal** (igual que regresión lineal)

$$
z = w^T x + b \tag{1}
$$

**Paso 2: Aplicar sigmoide** (la magia)

$$
\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}} = \frac{1}{1 + e^{-(w^T x + b)}} \tag{2}
$$

Interpretación: $\hat{y} = P(y=1|x; w, b)$ es la **probabilidad** de que el ejemplo pertenezca a la clase 1.

**Paso 3: Decisión**

$$
\text{Predicción} = \begin{cases}
1 & \text{si } \hat{y} \geq 0.5 \\
0 & \text{si } \hat{y} < 0.5
\end{cases} \tag{3}
$$

### ¿Por qué la Sigmoide?

Propiedades matemáticas importantes:

1. **Rango:** $\sigma(z) \in (0, 1)$ para todo $z \in \mathbb{R}$

2. **Simetría:** $\sigma(-z) = 1 - \sigma(z)$

3. **Derivada elegante:**
$$
\frac{d\sigma(z)}{dz} = \sigma(z)(1 - \sigma(z)) \tag{4}
$$

Esta derivada es **crucial** para gradient descent.

### Función de Pérdida: Binary Cross-Entropy

**¿Por qué no MSE?** El MSE con sigmoide crea una superficie no convexa (muchos mínimos locales).

**Solución:** Log-loss (Binary Cross-Entropy)

Para un solo ejemplo:

$$
\mathcal{L}(y, \hat{y}) = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})] \tag{5}
$$

**Intuición:**
- Si $y=1$: pérdida = $-\log(\hat{y})$ → queremos $\hat{y}$ cercano a 1
- Si $y=0$: pérdida = $-\log(1-\hat{y})$ → queremos $\hat{y}$ cercano a 0

Para todo el dataset:

$$
J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} \log(\hat{y}^{(i)}) + (1-y^{(i)}) \log(1-\hat{y}^{(i)})] \tag{6}
$$

### Gradientes para Gradient Descent

La derivada de la pérdida respecto a los pesos es:

$$
\begin{align}
\frac{\partial J}{\partial w_j} &= \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \cdot x_j^{(i)} \tag{7}\\
\frac{\partial J}{\partial b} &= \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \tag{8}
\end{align}
$$

**¡Sorpresa!** La forma es **idéntica** a regresión lineal, aunque la pérdida es diferente.

### Ejemplo Numérico

Supongamos 3 ejemplos de clasificación de tumores:

| Paciente | Tamaño (x) | Maligno (y) | z = 0.5x - 3 | σ(z) | Pérdida |
|----------|-----------|-------------|--------------|------|----------|
| 1 | 2 | 0 | -2.0 | 0.12 | 0.13 |
| 2 | 5 | 0 | -0.5 | 0.38 | 0.48 |
| 3 | 8 | 1 | 1.0 | 0.73 | 0.31 |

**Cálculo de pérdida para paciente 1:**
$$
\begin{align}
z &= 0.5(2) - 3 = -2.0 \\
\hat{y} &= \frac{1}{1 + e^{2.0}} = 0.12 \\
\mathcal{L} &= -[0 \cdot \log(0.12) + 1 \cdot \log(0.88)] = 0.13
\end{align}
$$

In [ ]:
# Verificación del ejemplo numérico
X_example = np.array([2, 5, 8])
y_example = np.array([0, 0, 1])
w, b = 0.5, -3.0

# Cálculos
z = w * X_example + b
y_pred = sigmoid(z)

# Pérdida para cada ejemplo
losses = -(y_example * np.log(y_pred) + (1 - y_example) * np.log(1 - y_pred))
avg_loss = np.mean(losses)

print("Verificación del Ejemplo Numérico:")
print("\nPaciente | x  | y | z     | σ(z)  | Pérdida")
print("-" * 50)
for i in range(len(X_example)):
    print(f"{i+1:8d} | {X_example[i]:2.0f} | {y_example[i]:1.0f} | {z[i]:5.2f} | {y_pred[i]:5.2f} | {losses[i]:7.3f}")
print("-" * 50)
print(f"Pérdida promedio: {avg_loss:.3f}")

### Interpretación Probabilística

La regresión logística asume que:

$$
P(y=1|x) = \sigma(w^T x + b) \tag{9}
$$

$$
P(y=0|x) = 1 - \sigma(w^T x + b) \tag{10}
$$

Podemos escribir ambas como:

$$
P(y|x) = (\hat{y})^y (1-\hat{y})^{1-y} \tag{11}
$$

La log-loss proviene de **maximizar la log-likelihood** (principio MLE).

---
## 💻 4. Implementación Desde Cero

In [ ]:
class RegresionLogistica:
    """
    Implementación desde cero de Regresión Logística para clasificación binaria.
    
    Parameters:
    -----------
    learning_rate : float
        Tasa de aprendizaje para gradient descent
    n_iterations : int
        Número de iteraciones de entrenamiento
    threshold : float
        Umbral de decisión (típicamente 0.5)
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, threshold=0.5):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.threshold = threshold
        
        # Parámetros del modelo
        self.weights = None
        self.bias = None
        
        # Tracking
        self.losses = []
    
    @staticmethod
    def sigmoid(z):
        """Función sigmoide con estabilidad numérica"""
        # Evitar overflow: si z es muy grande, e^(-z) es muy pequeño
        return np.where(
            z >= 0,
            1 / (1 + np.exp(-z)),
            np.exp(z) / (1 + np.exp(z))
        )
    
    def binary_cross_entropy(self, y_true, y_pred):
        """
        Calcula Binary Cross-Entropy Loss.
        
        L = -1/m * Σ[y*log(ŷ) + (1-y)*log(1-ŷ)]
        """
        # Clip para evitar log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        loss = -np.mean(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )
        return loss
    
    def fit(self, X, y):
        """
        Entrena el modelo usando gradient descent.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Features de entrenamiento
        y : np.ndarray, shape (n_samples,)
            Etiquetas binarias (0 o 1)
        """
        # Asegurar formato correcto
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        n_samples, n_features = X.shape
        
        # Inicialización de parámetros
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        
        print("🏃 Iniciando entrenamiento...\n")
        
        # Gradient descent
        for i in range(self.n_iter):
            # Forward pass
            # 1. Combinación lineal
            z = X @ self.weights + self.bias
            
            # 2. Aplicar sigmoide
            y_pred = self.sigmoid(z)
            
            # 3. Calcular pérdida
            loss = self.binary_cross_entropy(y, y_pred)
            self.losses.append(loss)
            
            # Backward pass: calcular gradientes
            # ∂L/∂w = (1/m) * X^T * (ŷ - y)
            # ∂L/∂b = (1/m) * Σ(ŷ - y)
            error = y_pred - y
            dw = (1 / n_samples) * (X.T @ error)
            db = (1 / n_samples) * np.sum(error)
            
            # Actualizar parámetros
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            # Logging
            if i % max(1, self.n_iter // 10) == 0:
                # Calcular accuracy en train
                y_pred_class = (y_pred >= self.threshold).astype(int)
                accuracy = np.mean(y_pred_class == y)
                print(f"Iter {i:4d} - Loss: {loss:.4f} - Accuracy: {accuracy:.4f}")
        
        print(f"\n✅ Entrenamiento completado")
        print(f"   Loss final: {self.losses[-1]:.4f}")
        
        return self
    
    def predict_proba(self, X):
        """
        Predice probabilidades.
        
        Returns:
        --------
        probabilities : np.ndarray, shape (n_samples,)
            P(y=1|x) para cada ejemplo
        """
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        z = X @ self.weights + self.bias
        return self.sigmoid(z)
    
    def predict(self, X):
        """
        Predice clases (0 o 1).
        
        Returns:
        --------
        predictions : np.ndarray, shape (n_samples,)
            Clase predicha para cada ejemplo
        """
        probabilities = self.predict_proba(X)
        return (probabilities >= self.threshold).astype(int)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase RegresionLogistica definida")

### Probemos nuestra implementación

In [ ]:
# Generar datos de clasificación binaria
X, y = generate_binary_classification(n_samples=500, n_features=2, random_state=42)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalizar features (importante para convergencia)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"📊 Datos:")
print(f"   Entrenamiento: {X_train.shape[0]} ejemplos")
print(f"   Prueba: {X_test.shape[0]} ejemplos")
print(f"   Features: {X_train.shape[1]}")
print(f"   Distribución de clases: {np.bincount(y_train)}")

In [ ]:
# Entrenar modelo
modelo = RegresionLogistica(learning_rate=0.1, n_iterations=1000)
modelo.fit(X_train, y_train)

In [ ]:
# Visualizar convergencia
fig = go.Figure()
fig.add_trace(go.Scatter(y=modelo.losses, mode='lines', name='Loss'))
fig.update_layout(
    title="Convergencia del Entrenamiento",
    xaxis_title="Iteración",
    yaxis_title="Binary Cross-Entropy Loss",
    template="plotly_white"
)
fig.show()

In [ ]:
# Evaluar modelo
y_pred_train = modelo.predict(X_train)
y_pred_test = modelo.predict(X_test)

y_proba_test = modelo.predict_proba(X_test)

# Calcular métricas
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("📊 Métricas de Evaluación:")
print("\n" + "="*50)
print(f"{'Métrica':<20} {'Train':<12} {'Test':<12}")
print("="*50)
print(f"{'Accuracy':<20} {accuracy_score(y_train, y_pred_train):<12.4f} {accuracy_score(y_test, y_pred_test):<12.4f}")
print(f"{'Precision':<20} {precision_score(y_train, y_pred_train):<12.4f} {precision_score(y_test, y_pred_test):<12.4f}")
print(f"{'Recall':<20} {recall_score(y_train, y_pred_train):<12.4f} {recall_score(y_test, y_pred_test):<12.4f}")
print(f"{'F1-Score':<20} {f1_score(y_train, y_pred_train):<12.4f} {f1_score(y_test, y_pred_test):<12.4f}")
print("="*50)

print("\n💡 Interpretación:")
print("   • Accuracy: % de predicciones correctas")
print("   • Precision: De los predichos como 1, cuántos son realmente 1")
print("   • Recall: De los que son 1, cuántos detectamos")
print("   • F1: Media armónica de precision y recall")

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_test)

fig = px.imshow(
    cm,
    labels=dict(x="Predicho", y="Real", color="Cantidad"),
    x=['Clase 0', 'Clase 1'],
    y=['Clase 0', 'Clase 1'],
    text_auto=True,
    color_continuous_scale='Blues'
)
fig.update_layout(title="Matriz de Confusión", template="plotly_white")
fig.show()

print("\n💡 Matriz de Confusión:")
print(f"   • TN (arriba-izq): Predijimos 0, era 0 ✓")
print(f"   • FP (arriba-der): Predijimos 1, era 0 ✗")
print(f"   • FN (abajo-izq): Predijimos 0, era 1 ✗")
print(f"   • TP (abajo-der): Predijimos 1, era 1 ✓")

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
roc_auc = auc(fpr, tpr)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'ROC (AUC = {roc_auc:.3f})',
    line=dict(color='blue', width=2)
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random (AUC = 0.5)',
    line=dict(color='red', dash='dash')
))
fig.update_layout(
    title="Curva ROC (Receiver Operating Characteristic)",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white"
)
fig.show()

print(f"\n💡 AUC (Area Under Curve) = {roc_auc:.3f}")
print("   • AUC = 1.0: Clasificador perfecto")
print("   • AUC = 0.5: Random (inútil)")
print("   • AUC > 0.8: Muy bueno")

---
## 🏭 5. Versión con Framework (Scikit-learn)

In [ ]:
# Entrenar con sklearn
sklearn_model = LogisticRegression(max_iter=1000)
sklearn_model.fit(X_train, y_train)

# Predicciones
y_pred_sklearn = sklearn_model.predict(X_test)
y_proba_sklearn = sklearn_model.predict_proba(X_test)[:, 1]

# Comparación
print("📊 Comparación: Nuestra Implementación vs Scikit-learn\n")
print("="*60)
print(f"{'Métrica':<20} {'Nuestra':<15} {'Scikit-learn':<15}")
print("="*60)
print(f"{'Accuracy':<20} {accuracy_score(y_test, y_pred_test):<15.4f} {accuracy_score(y_test, y_pred_sklearn):<15.4f}")
print(f"{'Precision':<20} {precision_score(y_test, y_pred_test):<15.4f} {precision_score(y_test, y_pred_sklearn):<15.4f}")
print(f"{'Recall':<20} {recall_score(y_test, y_pred_test):<15.4f} {recall_score(y_test, y_pred_sklearn):<15.4f}")
print(f"{'F1-Score':<20} {f1_score(y_test, y_pred_test):<15.4f} {f1_score(y_test, y_pred_sklearn):<15.4f}")

# AUC
from sklearn.metrics import roc_auc_score
auc_ours = roc_auc_score(y_test, y_proba_test)
auc_sklearn = roc_auc_score(y_test, y_proba_sklearn)
print(f"{'AUC':<20} {auc_ours:<15.4f} {auc_sklearn:<15.4f}")
print("="*60)

print("\n✅ ¡Resultados muy similares!")
print("\n💡 Ventajas de Scikit-learn:")
print("   • Regularización L1/L2 built-in")
print("   • Soporte para multi-class (one-vs-rest)")
print("   • Optimizaciones numéricas avanzadas")
print("   • Integración con pipelines y cross-validation")

---
## 🎯 6. Ejercicios Prácticos

### 🟢 Ejercicio 1: Clasificación de Diabetes

Usa el dataset de diabetes para predecir si un paciente tiene diabetes.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Entrenar un clasificador de diabetes
    
    Instrucciones:
    1. Carga el dataset 'diabetes' con load_dataset()
    2. Normaliza las features con StandardScaler
    3. Entrena un modelo RegresionLogistica
    4. Calcula accuracy, precision, recall y F1
    5. Retorna un dict con las métricas
    
    Returns:
    --------
    dict : {'accuracy': float, 'precision': float, 'recall': float, 'f1': float}
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# metricas = ejercicio_1()
# print("\n📊 Métricas obtenidas:")
# for k, v in metricas.items():
#     print(f"   {k}: {v:.4f}")

### 🟡 Ejercicio 2: Optimizar el Threshold

El threshold de 0.5 no siempre es óptimo. Encuentra el mejor threshold.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Encontrar el threshold óptimo
    
    Instrucciones:
    1. Entrena un modelo de regresión logística
    2. Obtén las probabilidades predichas en el test set
    3. Prueba diferentes thresholds: [0.3, 0.4, 0.5, 0.6, 0.7]
    4. Para cada threshold, calcula F1-score
    5. Retorna el threshold que maximiza F1
    
    Pista: y_pred = (y_proba >= threshold).astype(int)
    
    Returns:
    --------
    best_threshold : float
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# best_t = ejercicio_2()
# print(f"\n📊 Mejor threshold: {best_t}")

### 🔴 Ejercicio 3: Regresión Logística Regularizada

Implementa regularización L2 (Ridge) para prevenir overfitting.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Añadir regularización L2
    
    Instrucciones:
    1. Modifica la clase RegresionLogistica
    2. Añade término de regularización a la pérdida:
       J_reg = J + (λ/2m) * Σw²
    3. Modifica los gradientes:
       dw_reg = dw + (λ/m) * w
    4. Prueba con diferentes valores de λ: [0, 0.01, 0.1, 1.0]
    5. Compara accuracy en train vs test
    
    Returns:
    --------
    dict : {lambda: {'train_acc': float, 'test_acc': float}}
    """
    # TODO: Tu código aquí
    # class RegresionLogisticaRegularizada(RegresionLogistica):
    #     def __init__(self, ..., regularization=0.01):
    #         ...
    
    pass

# Descomentar para probar
# results = ejercicio_3()
# print("\n📊 Efecto de la regularización:")
# for lam, metrics in results.items():
#     print(f"   λ={lam}: Train={metrics['train_acc']:.3f}, Test={metrics['test_acc']:.3f}")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **Regresión Logística** es el algoritmo fundamental para clasificación binaria

2. **La función sigmoide** $\sigma(z) = \frac{1}{1 + e^{-z}}$ convierte valores reales en probabilidades (0,1)

3. **Binary Cross-Entropy** es la función de pérdida apropiada:
   $$\mathcal{L} = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

4. **Interpretación probabilística**: $\hat{y} = P(y=1|x)$

5. **Métricas de clasificación**:
   - **Accuracy**: % correcto (cuidado con clases desbalanceadas)
   - **Precision**: De los positivos predichos, cuántos son reales
   - **Recall**: De los positivos reales, cuántos detectamos
   - **F1**: Balance entre precision y recall
   - **AUC-ROC**: Mide capacidad de discriminación

6. **El threshold importa**: 0.5 es default pero puede no ser óptimo

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"Logistic Regression"** - David Cox (1958)
  - Paper original que introduce regresión logística
  - Contexto: Usado originalmente en estadística biomédica

- **"The Origins of Logistic Regression"** - Cramer (2002)
  - Excelente perspectiva histórica

#### 📖 Recursos Educativos

- **StatQuest - Logistic Regression**
  - [YouTube Video](https://www.youtube.com/watch?v=yIYKR4sgzI8)
  - Explicación clara y visual

- **Pattern Recognition and Machine Learning** - Bishop
  - Capítulo 4.3: Probabilistic Discriminative Models

#### 💻 Herramientas

- [Scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Scikit-learn: Classification Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)

### 🤔 Preguntas para Reflexionar

1. **¿Por qué se llama "regresión" si es para clasificación?**
   - Razones históricas: regresamos la probabilidad (valor continuo)
   - Luego aplicamos un threshold para clasificar

2. **¿Cuándo usar accuracy vs F1-score?**
   - Accuracy: Cuando las clases están balanceadas
   - F1: Cuando hay desbalance o ambos errores (FP y FN) importan

3. **¿Qué hacer con clases desbalanceadas (ej: 95% clase 0, 5% clase 1)?**
   - Ajustar threshold
   - Usar class weights
   - Oversampling (SMOTE) / Undersampling
   - Métricas: Precision-Recall curve, F1, AUC-PR

### 📊 Cheat Sheet: ¿Qué métrica usar?

| Escenario | Métrica Principal | Razón |
|-----------|-------------------|-------|
| Clases balanceadas | **Accuracy** | Simple y efectiva |
| Clases desbalanceadas | **F1-Score** | Balance precision/recall |
| Minimizar falsos positivos | **Precision** | Ej: spam detection |
| Minimizar falsos negativos | **Recall** | Ej: detección cáncer |
| Comparar modelos | **AUC-ROC** | Independiente de threshold |

---

## ➡️ Próximo Paso

En el siguiente notebook, **04. Regresión Softmax**, extenderemos la clasificación binaria a **múltiples clases**:

- Generalizar sigmoide a softmax
- Codificación one-hot de etiquetas
- Categorical Cross-Entropy
- Clasificación multi-clase (ej: dígitos 0-9)
- Estrategias one-vs-rest y one-vs-one

---

<div align="center">

**🎉 ¡Has dominado la clasificación binaria! 🎉**

**Continúa con: [04. Regresión Softmax](04-regresion-softmax.ipynb)**

[← 02. Gradient Descent](02-gradient-descent.ipynb) | [Índice de ML Clásico](README.md) | [04. Regresión Softmax →](04-regresion-softmax.ipynb)

</div>